In [322]:
import sys
sys.path.append("/Users/keisuke/Documents/projects/todo/worms/python/rmsKit")
import os
import torch
from lattice import BLBQ
# # os.environ["CUDA_VISIBLE_DEVICES"] = ""
# # os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import rms_torch
import numpy as np
# u = np.load("/home/user/project/python/rmsKit/array/KH/3site/sel/Jx_1_Jy_1_Jz_1_hx_0_hz_0/M_1/u/0.npy")
path = "/Users/keisuke/Documents/projects/todo/worms/python/rmsKit/array/torch/BLBQ1D_loc/J0_1_J1_0.3333_hx_0_hz_0/1_mel/radam/lr_0.01_epoch_5000/loss_0.0000457/u/0.npy"

u = np.load(path)
# u = u.T
# u = np.random.randn(3, 3)
# u,s,v = np.linalg.svd(u)
# u = u@v
np.save(path, np.ascontiguousarray(u.T))
# u = torch.eye(3).to(torch.float64)

In [329]:

u = torch.tensor(u)
# device = torch.device("cuda")
p = dict(
    J0 = 1,
    J1 = 0.1,
    hx=0,
    hz=0,
    lt = 1,
    obc = True,
)

# kron product of u for 6 times
U = u
for i in range(3):
    U = torch.kron(U, u)

H = BLBQ.system([4], p)[0]
H = U@H@U.T.conj()
h_loc = BLBQ.local(p)
# Create an instance of the CustomModel class
E, V = np.linalg.eigh(H)
off = np.ones(H.shape[0]) * 100
off = np.diag(off)
H_abs = H - off
H_abs = -np.abs(H_abs)
H_abs = H_abs + off
E_abs, V_abs = np.linalg.eigh(H_abs)

U.shape

/var/folders/h6/fvcv1prn2vd277wly2ssj0ch0000gn/T/ipykernel_35859/3115941703.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u = torch.tensor(u)


torch.Size([81, 81])

In [330]:
U2 = torch.kron(u, u)
h = U2@(h_loc[0][0])@U2.T.conj()
off = np.ones(h.shape[0]) * 100
off = np.diag(off)
h_abs = h - off
h_abs = -np.abs(h_abs)
h_abs = h_abs + off
e_abs, v_abs = np.linalg.eigh(h_abs)

e, v  = np.linalg.eigh(h)



In [331]:
e_abs - e

array([-1.28261879e-08,  1.28882915e-09,  1.60265312e-08,  3.55271368e-15,
        1.99840144e-15,  4.21884749e-15, -3.64530850e-09, -8.43853876e-10,
        3.10862447e-15])

In [333]:
T = 1
z = np.exp(-E / T)
# Calculate average energy
e_exp = (E * z).sum() / z.sum()
# Calculate average energy squared
e_squared_exp = (E**2 * z).sum() / z.sum()
# Calculate specific heat (C_v = (⟨E²⟩ - ⟨E⟩²)/T²)
c_v = (e_squared_exp - e_exp**2) / (T**2)

z_abs = np.exp(-E_abs / T)

AS = z.sum()/z_abs.sum()

print(AS, e_exp / 4, c_v / 4)



0.999999993788288 -0.6273598713985127 0.4040438312761516


In [327]:
e_exp, AS

(-0.9641770245269339, 0.9999999971720758)